# Glycosylation occupancy benchmark — GPU runner

Runs the two model-dependent stages of `analysis/experimental_glycosylation_sites`
on a Colab GPU, for **both** models:

| Model | Scores | Designs |
|---|---|---|
| ProteinMPNN `v_48_020` | conditional, 8 decoding orders | 32 designs, T=0.1 |
| ESM-IF1 `esm_if1_gvp4_t16_142M_UR50` | teacher-forced prefix conditional | 32 designs, T=0.1 |

Everything downstream of scoring — matching, contrasts, the cluster bootstrap,
significance testing, the figures — is model-agnostic and stays on your laptop.
This notebook only produces `results/scores/*` and `results/designs/*`.

### Two things to know before you read any number out of this

**The ProteinMPNN alphabet was corrected on 2026-08-20.** `mpnn_scoring.ALPHABET`
had held `ARNDCQEGHILKMFPSTWYVX`, which is a three-letter lookup table from
inside `parse_PDB_biounits`, not the model's token alphabet. The real one is
`ACDEFGHIKLMNPQRSTVWYX`. Consequence: `p_asn_at_n` was reading P(aspartate), and
designed sequences were decoded with the wrong letters. P(Ser) and P(Thr) were
correct by coincidence. **Any ProteinMPNN score or retention figure produced
before that date needs regenerating** — which is half of why this notebook runs
ProteinMPNN too, not just ESM-IF. Cell 6 re-verifies the alphabet against
ProteinMPNN's own source before anything is scored.

**ESM-IF's conditional is not ProteinMPNN's.** ESM-IF is autoregressive, so it
gives P(residue | backbone, native prefix) from one deterministic pass — no
decoding-order distribution, and no C-terminal context. ProteinMPNN conditions
on the whole rest of the sequence and averages over 8 sampled orders. Compare
the two models' *matched-pair contrasts*, never their raw score magnitudes.

## 1. Check the GPU

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU — set Runtime > Change runtime type > GPU")
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Install dependencies

ESM-IF needs `torch-geometric` **and** `torch-scatter` (its GVP encoder imports
both), but **not** `torch-sparse` — which is the one that reliably fails to
build. `torch-scatter` comes from the PyG wheel index matched to this runtime's
exact torch build; installing it from PyPI would try to compile it.

In [ ]:
import torch, re, subprocess, sys

def pip(*a):
    print("$ pip", *a)
    subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)

tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".","") if torch.cuda.is_available() and torch.version.cuda else "cpu"
index = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"
print("PyG wheel index:", index)

pip("fair-esm==2.0.0")
pip("torch-geometric")
try:
    pip("torch-scatter","-f",index)
except subprocess.CalledProcessError:
    print("!! torch-scatter wheel unavailable for this torch/cuda combination.")
    print("   Pick an older torch runtime or check", index)
pip("biotite","pandas","scipy","biopython")

## 3. Mount Drive and point at the bundle

Build the bundle once on your laptop:

```bash
cd analysis/experimental_glycosylation_sites
python pipeline/30_package_for_colab.py --out results/colab_bundle --tar
```

then upload `results/colab_bundle.tar` to Drive. It holds only the ~900
structures these stages actually open (~1.2 GB raw, ~250 MB gzipped), plus the
manifests and matched-pair tables.

Results are written **straight to Drive** so a disconnected runtime costs you
time and nothing else — every stage is resumable and skips what is already done.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BUNDLE_TAR = Path('/content/drive/MyDrive/sugarfix/colab_bundle.tar')   # <-- edit
RESULTS    = Path('/content/drive/MyDrive/sugarfix/results')            # <-- edit
RESULTS.mkdir(parents=True, exist_ok=True)
print("bundle exists:", BUNDLE_TAR.exists(), "|", BUNDLE_TAR)
print("results ->", RESULTS)

## 4. Get the code

Public clone works as-is. For a private repo, generate a fine-grained PAT with
read access to the repo and paste it when prompted — it is read via `getpass`,
so it is not echoed and does not land in the saved notebook.

In [ ]:
import os, subprocess, getpass
from pathlib import Path

REPO   = "github.com/LBDillon/SugarFix.git"
BRANCH = "main"          # <-- the branch carrying the ESM-IF adapter
PRIVATE = True           # set False for an anonymous clone

if not Path('/content/SugarFix').exists():
    url = f"https://{REPO}"
    if PRIVATE:
        token = getpass.getpass("GitHub token (input hidden): ").strip()
        url = f"https://{token}@{REPO}"
    subprocess.run(["git","clone","--depth","1","-b",BRANCH,url,"/content/SugarFix"], check=True)

MODULE = Path('/content/SugarFix/analysis/experimental_glycosylation_sites')
os.chdir(MODULE); print("cwd:", os.getcwd())

# ProteinMPNN is a separate public repo; its weights ship inside it.
if not Path('/content/ProteinMPNN').exists():
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/dauparas/ProteinMPNN.git","/content/ProteinMPNN"], check=True)
print("MPNN weights:", len(list(Path('/content/ProteinMPNN/vanilla_model_weights').glob('*.pt'))), "checkpoints")

## 5. Unpack the bundle

Structures are gunzipped into `data/cache/pdb/`, which is the first directory
`runner_support.structure_paths()` searches — so the pipeline finds them with no
path arguments at all.

In [ ]:
import tarfile, gzip, shutil, time
from pathlib import Path

work = Path('/content/bundle')
if not work.exists():
    work.mkdir(parents=True)
    with tarfile.open(BUNDLE_TAR) as tar:
        tar.extractall(work)

pdb_dir = MODULE/'data'/'cache'/'pdb'
pdb_dir.mkdir(parents=True, exist_ok=True)
src = work/'structures'
gz = list(src.glob('*.gz')); plain = [p for p in src.iterdir() if p.suffix in ('.pdb','.cif')]
t0 = time.time()
for i, p in enumerate(gz, 1):
    target = pdb_dir/p.stem
    if not target.exists():
        with gzip.open(p,'rb') as fh, open(target,'wb') as out: shutil.copyfileobj(fh, out)
    if i % 300 == 0: print(f"  {i}/{len(gz)} ({time.time()-t0:.0f}s)")
for p in plain:
    if not (pdb_dir/p.name).exists(): shutil.copyfile(p, pdb_dir/p.name)

for sub in ('manifests','matching'):
    dest = MODULE/'results'/sub; dest.mkdir(parents=True, exist_ok=True)
    for p in (work/sub).glob('*.csv'):
        if not (dest/p.name).exists(): shutil.copyfile(p, dest/p.name)

print(f"structures: {len(list(pdb_dir.glob('*')))}")
print(f"manifests : {len(list((MODULE/'results'/'manifests').glob('*.csv')))}")
print((work/'BUNDLE.json').read_text())

## 6. Preflight — verify before computing anything

Three checks, each guarding a defect that has actually happened in this project:

1. both adapters satisfy the declared protocols;
2. **ProteinMPNN's alphabet decodes its own `S` tensor back to the native
   sequence** — the check that would have caught the alphabet bug immediately;
3. ESM-IF's residue indexing agrees with the manifest's on a real chain.

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK','TRUE')
sys.path.insert(0,'src'); sys.path.insert(0,'/content/ProteinMPNN')
import pandas as pd, numpy as np

from experimental_glycosylation_sites import adapters
from experimental_glycosylation_sites.adapters.base import SequonScorer, SequenceDesigner
for name in adapters.available():
    a = adapters.load(name) if name!='proteinmpnn' else adapters.load(name, proteinmpnn_dir='/content/ProteinMPNN')
    print(f"{name:12s} SequonScorer={isinstance(a,SequonScorer)} SequenceDesigner={isinstance(a,SequenceDesigner)} {a.describe()}")

# --- the alphabet check ---
from protein_mpnn_utils import parse_PDB, StructureDatasetPDB, tied_featurize
from experimental_glycosylation_sites.mpnn_scoring import ALPHABET
from experimental_glycosylation_sites.runner_support import structure_paths
paths = structure_paths()
man = pd.read_csv('results/manifests/candidate_manifest_dataset.csv', low_memory=False)
ok = tot = 0
for (pdb, chain), _ in list(man.groupby(['structure_pdb_id','structure_chain_id']))[:5]:
    p = paths.get(str(pdb).upper())
    if p is None: continue
    prot = StructureDatasetPDB(parse_PDB(str(p), input_chain_list=[str(chain)]), truncate=None, max_length=20000)[0]
    S = tied_featurize([prot],'cpu',{prot['name']:([str(chain)],[])},None,None,None,None,None,ca_only=False)[1]
    dec = ''.join(ALPHABET[i] for i in S[0].numpy())
    n = min(len(dec), len(prot['seq']))
    ok += sum(1 for i in range(n) if dec[i]==prot['seq'][i]); tot += n
pct = 100*ok/tot
print(f"\nALPHABET = {ALPHABET}")
print(f"decodes ProteinMPNN's own S tensor to the native sequence: {pct:.2f}%")
assert pct > 95, f"ALPHABET is wrong ({pct:.1f}%) — do not score with it"
print("alphabet OK")

# --- ESM-IF index agreement ---
from experimental_glycosylation_sites import esmif_scoring as E
checked = 0
for r in man.itertuples(index=False):
    p = paths.get(str(r.structure_pdb_id).upper())
    if p is None: continue
    try: m = E.chain_mapping(p, r.structure_chain_id, str(r.structure_pdb_id))
    except E.ChainUnreadableError: continue
    idx = (int(r.n_model_index), int(r.plus1_model_index), int(r.plus2_model_index))
    m.check_triplet(m.map_indices(idx), r.triplet); checked += 1
    if checked >= 25: break
print(f"ESM-IF index mapping verified on {checked} sequons")

## 7. What is left to do, and what is already done

Done on the laptop, already corrected, **do not redo**:

| File | Sites |
|---|---|
| `scores_dataset_alphabet_corrected.csv` | 342 |
| `scores_controls_alphabet_corrected.csv` | 558 |
| `scores_secretory_alphabet_corrected.csv` | 262 |

Left, and the reason this session exists:

1. **ESM-IF scoreability and scores** — the second model, never yet run.
2. **Retention for both models** — ~13.6 h of CPU per model, which is why it is
   here. Every ProteinMPNN retention number is invalid: designs were decoded with
   the wrong alphabet, so `classify_retention`'s asparagine test could not fire.

Retention runs on `scoring_manifest.csv` (1,725 chains) and
`manifest_matched_secretory.csv` (233 chains) — those are the two sources
`10_analyse_retention_by_class.py` reads, not the candidate pools.

### 7a. ESM-IF scoreability

Its parser is biotite, not Biopython, so which sites it can address is its own question.

In [ ]:
import subprocess, sys, time

SCORE_SETS = [('dataset',   'candidate_manifest_dataset'),
              ('controls',  'manifest_matched_controls'),
              ('secretory', 'manifest_matched_secretory')]
DESIGN_SETS = [('scoring_manifest', 'scoring_manifest'),
               ('secretory',        'manifest_matched_secretory')]

def run(*args):
    t0 = time.time(); print("$", " ".join(str(a) for a in args), flush=True)
    r = subprocess.run([sys.executable, *[str(a) for a in args]], text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-2500:]); print(f"[{(time.time()-t0)/60:.1f} min]\n", flush=True)
    if r.returncode: raise RuntimeError(f"FAILED: {args}")

for tag, manifest in SCORE_SETS:
    run('pipeline/05_scoreability.py',
        f'results/manifests/{manifest}.csv',
        RESULTS/f'scoreability_{tag}_esm_if.csv',
        '--model','esm_if','--device','cuda')

### 7b. ESM-IF scores

One teacher-forced pass per chain — the cheap half. ~1,000 chains total.

In [ ]:
for tag, manifest in SCORE_SETS:
    run('pipeline/07_score.py',
        f'results/manifests/{manifest}.csv',
        RESULTS/f'scores_{tag}_esm_if.csv',
        '--model','esm_if','--device','cuda')

### 7c. Retention — the expensive half

Start with the secretory set (233 chains). **Read its wall time before launching
`scoring_manifest`, which is 7.4x larger.** Both stages resume, so a dropped
runtime costs only the chain in flight.

In [ ]:
for tag, manifest in DESIGN_SETS:
    run('pipeline/08_design.py',
        f'results/manifests/{manifest}.csv',
        RESULTS/f'retention_{tag}_esm_if.csv',
        '--model','esm_if','--device','cuda')

In [ ]:
# ProteinMPNN retention, with the corrected alphabet. Supersedes
# mpnn_retention_frozen_2026-08-18.csv and mpnn_retention_secretory.csv entirely.
for tag, manifest in DESIGN_SETS:
    run('pipeline/08_design.py',
        f'results/manifests/{manifest}.csv',
        RESULTS/f'retention_{tag}_proteinmpnn_corrected.csv',
        '--model','proteinmpnn','--device','cuda')

## 8. What came back

In [ ]:
import pandas as pd
for f in sorted(RESULTS.glob('*.csv')):
    try:
        d = pd.read_csv(f, low_memory=False); extra = ''
        if 'conditional_sequon_score' in d:
            extra = f"  mean score {d.conditional_sequon_score.mean():+.3f}"
        if 'std_frac_full_sequon_retained' in d:
            extra = f"  mean retention {d.std_frac_full_sequon_retained.mean():.3f}"
        if 'scoreable' in d:
            extra = f"  scoreable {int(d.scoreable.sum())}/{len(d)}"
        print(f"{f.name:50s} {len(d):6d} rows{extra}")
    except Exception as e:
        print(f"{f.name:50s} unreadable: {e}")

## Back on the laptop

Copy the files down from Drive, then point the retention stage at the corrected
designs — `10_analyse_retention_by_class.py` has its `SOURCES` **hard-coded** to
the old filenames, so it will silently keep using the invalid ones otherwise:

```python
SOURCES = [
    ("results/designs/retention_scoring_manifest_proteinmpnn_corrected.csv",
     "results/manifests/scoring_manifest.csv"),
    ("results/designs/retention_secretory_proteinmpnn_corrected.csv",
     "results/manifests/manifest_matched_secretory.csv"),
]
```

Then:

```bash
python pipeline/09_analyse_scores.py optimal      # PRIMARY
python pipeline/09_analyse_scores.py secretory
python pipeline/10_analyse_retention_by_class.py
python pipeline/10b_analyse_retention_paired.py
python pipeline/11_significance.py                # all 8 tests, corrected
```

`09_analyse_scores.py` also hard-codes `scores_dataset.csv` / `scores_controls.csv`
/ `scores_secretory.csv`. Either rename the corrected files over them (keep the
originals) or edit the paths — otherwise the corrected scores are computed and
then not used.

### The score result you already have

Corrected, laptop, scores only — retention still outstanding:

| Comparison | Old (SD) | Corrected (SD) | Verdict |
|---|---|---|---|
| optimal (PRIMARY) | +0.458 | **+0.640** | inconclusive |
| secretory | +0.073 | **+0.090** | inconclusive |
| bacterial (diagnostic) | −0.157 | **−0.396** | beyond the margin, p = 2.7e−06 |
| cytosolic (diagnostic) | +0.067 | **−0.028** | inconclusive |

The null on the biological question stands. The bacterial diagnostic firing hard
and negative is a **confounding detector working**, not support for the
hypothesis — it says structural matching never removed a taxonomy/composition
difference, so that set cannot serve as a control. The old alphabet was masking
it at p = 0.063.